# Preliminaries -- Notebook Setup

In [ ]:
# Begin - startup boilerplate code

import pkgutil

if 'fibertree_bootstrap' not in [pkg.name for pkg in pkgutil.iter_modules()]:
  !pip  install git+https://github.com/Fibertree-project/fibertree-bootstrap --quiet

# End - startup boilerplate code

from fibertree_bootstrap import *

fibertree_bootstrap(style="uncompressed", animation="movie")

In [ ]:
A_dense = [ [ 0, 1, 0, 2, 0, 0 ],
               [ 3, 0, 4, 5, 6, 0],
               [ 2, 0, 0, 7, 0, 9],
               [ 0, 0, 10, 0, 3, 4],
               [ 9, 3, 1, 7, 0, 2],
               [ 5, 2, 0, 3, 4, 0]]
A = Tensor.fromUncompressed(["M", "N"], A_dense)
A.setName("A")
A.setColor("green")
M = A.getShape("M")
N = A.getShape("N")

# Let's focus on a one-rank fiber
a_m = A.getRoot()

B = Tensor.fromFiber(rank_ids="D", fiber=a_m.getPayload(4))
B.setName("B")
displayTensor(B)

# Outline
1. Terminology
2. Example Populate Implementation
3. Examples

# Terminology
### Let us begin with some definitions
- Map temp --> Reduce Temp --> Populate Temp/Final output
- reduce temp has the shape of the iteration space after all contracted rank variables have been removed (note that this is different from Map)

The populate action takes the following as input *at a given point in the reduced iteration space*:
  1. The reduce temp payload
  2. The reduce temp coordinate
  3. The current output fiber *for the mutable rank in consideration*
  4. A coordinate operator (user-defined)
  5. A compute operator (user-defined)

The populate action outputs:
  1. A modified output fiber *for the mutable rank in consideration*

Populate consists of two operators:
  1. The coordinate operator
  2. The compute operator

The coordinate operator takes as input:
  1. The reduce temp payload
  2. The reduce temp coordinate
  3. The current output fiber *for the mutable rank in consideration*

The coordinate operator outputs:
  1. A set of valid coordinates to be updated
  2. A set of dead coordinates to be deleted

  Note: the coordinate operator does NOT modify anything. We are not allowing populate to reduce.
  
  The populate action will process the output of coordinate and compute and perform the modifications.



The compute operator takes as input:
  1. A coordinate from the coordinate operator
  2. The reduce temp payload  
  Populate calls the compute operator for every valid coordinate from the coordinate operator.

The compute operator outputs:
   1. A (modified) payload. (per call)

Question:
  1. are we enforcing that populate MUST be at the innermost level of the loop nest?
    - leaf fiber --> implementation artifact
      - talking about a fiber does not require that that fiber be a leaf fiber
      - populate operates on a *mathematical* fiber where the payloads are always leaf payloads

EDGE 2 revisions (08/27/2024):
  - If we're using the default populate, the output is exactly the reduce temporary.
  - clarifying that the recipe works point-by-point
    - map, reduce, optional populate if reduction done (note we only iterate through once)
  -


  - why not allow the user to provide an initial state, modify it at will, etc..., but the final thing must be a valid fiber
    - this an implementation and doesn't followe the ODE.

***
# Example Implementation
#### Let us look at an implementation of populate.

In [ ]:
def populate(lhs_fiber, rhs_coord, rhs_val, coord_op, compute_op):

  # The coordinate operator takes as input:
  #    The RHS coordinate (rhs_coord)
  #    The corresponding data value on the RHS (rhs_val). Note that this is
  #       also the reduce temp.
  #    An lhs_fiber
  # It returns:
  #   - A valid set of coordinates for the output
  #   - A set of "dead" coordinates for the output
  # The coord_op does not mutate the lhs_fiber (Read only arguments)
  valid_coords, dead_coords = coord_op(lhs_fiber, rhs_coord, rhs_val)

  # The compute operator takes as input:
  #   1. A coordinate from the coordinate operator
  #   2. The reduce temp payload
  #   Populate calls the compute operator for every valid coordinate from
  #   the coordinate operator.

  # The compute operator outputs:
  #    1. A (modified) payload. (per call)
  # New proposal:
  for coord in valid_coords:
    lhs_ref = lhs_fiber.getPayloadRef(coord)
    lhs_ref <<= compute_op(coord, rhs_val)

  # coord is telling compute_op which coordinate is changing


  # And delete our dead coords

  #for coord, z_val in lhs_fiber:
  for coord in dead_coords:
      lhs_ref = lhs_fiber.getPayloadRef(coord)
      # hard vs soft empty is implementation dependent
      lhs_ref <<= lhs_fiber.getDefault()

  #lhs_fiber is mutable, so no need to return anything

In [ ]:

## WARNING! This is a super naive implementation of populate.

#################################
# Default Coordinate Operator
#################################
def pass_through_coord(lhs_fiber, rhs_coord, rhs_val):
    #valid_coords = lhs_fiber.getCoords()
    valid_coords = [rhs_coord]
    dead_coords = []
    return valid_coords, dead_coords

#################################
# Default Compute Operator
#################################
def pass_through_payload(coord, rhs_val):
  return rhs_val

def compact_coord(lhs_fiber, rhs_coord, rhs_val):
    curr_coords = lhs_fiber.getCoords() #returns nonZero coordiantes in lhs

    # Get the last populated coordinate
    if len(curr_coords) == 0:
      last_coord = -1
    else:
      last_coord = curr_coords[-1]
    # our new valid coord is next to the last popuplated thing in the lhs
    valid_coords = [last_coord + 1]
    dead_coords = []
    return valid_coords, dead_coords

########################
# Coordinate Operators
########################
# Coord should only return active coords
# Then create the active vals in populate
def largestVal3(z_fiber, A_coord, A_val):
    z_fiber_as_list = []
    coords = []
    vals = []
    z_coords = []

    # Gather coordinates and values in Z
    for star_coord, z_val in z_fiber:
      z_fiber_as_list.append((star_coord, z_val.value))
      coords.append(star_coord)
      z_coords.append(star_coord)
      vals.append(z_val.value)

    # z_coords = copy.deepcopy(coords)

    # Now include the RHS coordinate and value
    coords.append(A_coord)
    vals.append(A_val.value)


    # get the top three values  and their corresponding coordinates
    top_3 = sorted(zip(vals, coords), reverse=True)[:3]

    live_coords = [val_coord[1] for val_coord in top_3]

    # Which coordinates are dead?
    dead_coords = list(set(z_coords) - set(live_coords))

    # Did we update any coordinates?
    valid_coords = list(set(live_coords) - set(z_coords))

    print(len(valid_coords))

    return valid_coords, dead_coords

#
# Note: this breaks our definition of the coordinate operator!
#      since we are modifying coordinates in Z that are also A_vals.
# #
# def valAsCoord(z_fiber, A_coord, A_val):
#     z_fiber_as_list = []
#     coords = []
#     vals = []

#     is_set = False
#     for star_coord, z_val in z_fiber:
#       # first, check that Z does not already have A_val as a coordainte
#       # if it does, replace it with A_val -- this is where we break the definition of the coordinate operator
#       if star_coord == A_val:
#         coords.append(star_coord)
#         vals.append(A_val.value)
#         is_set = True
#       else:
#         coords.append(star_coord)
#         vals.append(Z_val.value)

#     if not is_set:
#       # make the value a coordinate
#       coords.append(A_val.value)
#       vals.append(A_val.value)

#     return coords, vals



## Proposal for Multi-rank Populate

```python
def populate(lhs_fiber, rhs_coord, rhs_val, coord_op, compute_op):
```

- a *fiber* is the set of coordinates and payloads of a particular rank, all other ranks fixed.
- extension to multi-ranks: simply allow multiple ranks to be "unfixed"?

```python
def populate(lhs_fiber, rhs_coord_tuple, rhs_val, coord_op, compute_op):
```

Example: Find the largest three values in each $m$ slice:
$$
Z_{m, n*, k*} = A_{m, n, k} :: \lll_{n^*, k^*} \mathbf{1}(\text{max-val-3})
$$

- in this case, populate would look at the entire n-k fiber (is that the right term?), the current (m, n, k) coordinate for the RHS, the corresponding RHS value, and apply the max-val-3 coordinate operator to these three inputs.
- recall we assume $Z$ is alwasy initialized to the empty value.
- Any issues with this approach? (Seems like a straightforward implementation to me).


***
***

In [ ]:
Z = Tensor(rank_ids=["M", "N*", "K*"])
Z.setName("Z")
#Z.setShape([B.getShape("D")])
#Z.setDefault(None)
a_d = B.getRoot()
z_d = Z.getRoot()

# N = A.getShape("N")

# canvas = createCanvas(A, Z)

# Let's walk the iteration space:
for d, (a_val) in a_d:
  ######################################
  # Now, this is the input to populate:
  ######################################
  # Get the current d fiber of Z -- we're 1D so it is just z_d
  populate(z_d, d, a_val, coord_op=largestVal3, compute_op=pass_through_payload)


  displayTensor(z_d)

print("***********************************")
displayTensor(B)
displayTensor(Z)

# Original Examples


***
## Example 1: Return the Largest three Values
\begin{align}
Z_{d^*} &= A_d :: \lll_{d^*} \mathbf{1}(\text{max-val-3})
\end{align}

"For each $𝑑^∗$ point in Z, check if it contains one of the largest three values of all
values seen thus far. Remove any values that are not in the largest three values."


In [ ]:
Z = Tensor(rank_ids=["D*"])
Z.setName("Z")
#Z.setShape([B.getShape("D")])
#Z.setDefault(None)
a_d = B.getRoot()
z_d = Z.getRoot()

# N = A.getShape("N")

# canvas = createCanvas(A, Z)

# Let's walk the iteration space:
for d, (a_val) in a_d:
  ######################################
  # Now, this is the input to populate:
  ######################################
  # Get the current d fiber of Z -- we're 1D so it is just z_d
  populate(z_d, d, a_val, coord_op=largestVal3, compute_op=pass_through_payload)


  displayTensor(z_d)

print("***********************************")
displayTensor(B)
displayTensor(Z)

In [ ]:
## Compact Z
## Y[\rho(d)] = Z_d --> we can use populate!

Y = Tensor(rank_ids=["D*"])
Y.setName("Y")

z_d = Z.getRoot()
y_d = Y.getRoot()

# Let's walk the iteration space:
for d, (z_val) in z_d:
  ######################################
  # Now, this is the input to populate:
  ######################################
  # Get the current d fiber of Z -- we're 1D so it is just z_d
  populate(y_d, d, z_val, coord_op=compact_coord, compute_op=pass_through_payload)


  displayTensor(y_d)

print("***********************************")
displayTensor(Z)
displayTensor(Y)

***
## Example 2: Coordinate as Value

\begin{align}
Z_{m, n, c*} = A_{m, n} :: \lll_{c*} \text{pass-through}(ValAsCoord)
↔\\
Z_{m, n, A_{m, n}} = A_{m, n}
\end{align}





In [ ]:
def ValAsCoord(z_fiber, A_coord, A_val):
    z_fiber_as_list = []
    live_coords = []
    live_vals = []
    z_coords = []

    is_set = False
    for star_coord, z_val in z_fiber:
      # first, check that Z does not already have A_val as a coordainte
      # if it does, replace it with A_val
      z_coords.append(star_coord)
      if star_coord == A_val:
        live_coords.append(star_coord)
        live_vals.append(A_val.value)
        is_set = True
      else:
        live_coords.append(star_coord)
        live_vals.append(z_val.value)

    if not is_set:
      # make the value a coordinate
      live_coords.append(A_val.value)
      live_vals.append(A_val.value)

    # coords in z, but not in the live set
    dead_coords = list(set(z_coords) - set(live_coords))

    # Did we update any coordinates?
    # valid_coords = (A_val) -- set consisting of A_val
    # valid_coords = set(A_val)
    valid_coords =    list(set(live_coords) - set(z_coords))
    return valid_coords, dead_coords

## When presenting this - make sure to remove the live_coords
def ValAsCoord(z_fiber, A_coord, A_val):
  valid_coords = list(set(A_val))
  dead_coords = []
  return valid_coords, dead_coords


## Example 2b: What if we wanted to use A to index into Z, and set the corresponding value to B?
With a mask:
\begin{align}
T_{m, n, c*} = A_{m, n}  :: \lll_{c*} 1(ValAsCoord) \\
Z_{m, n, p} = B_{m, n} \cdot T_{m, n, p}\\
\end{align}

We don't want value-based computation in the subscripts:
$$
Z_{m, n, A_{m, n}} = B_{m, n}
$$



In [ ]:
Z = Tensor(rank_ids=["M", "N", "D*"])
Z.setName("Z")
#print(A.getShape())
#Z.setShape(A.getShape()+ A.get)
displayTensor(Z)
#Z.setDefault(None)
a_m = A.getRoot()
z_m = Z.getRoot()

# N = A.getShape("N")

# canvas = createCanvas(A, Z)

# Let's walk the iteration space:
for m, (z_n, a_n) in z_m << a_m:
  for n, (z_d, a_val) in z_n << a_n:
    ######################################
    # Now, this is the input to populate:
    ######################################
    # Get the current d fiber of Z -- we're 1D so it is just z_d
    populate(z_d, d, a_val, coord_op=ValAsCoord, compute_op=pass_through_payload)


print("***********************************")
displayTensor(A)
displayTensor(Z)


Or with tuples:
\begin{align}
Z_{m, n, c*} = (A_{m, n} \cdot B_{m,n}) :: \bigwedge \text{tuple} \lll_{c*} second(first) \\
\\
Z_{m, n, A_{m, n}} = B_{m, n}
\end{align}

***
## Example 3: Coordinate as Value -- set value to 1

\begin{align}
Z_{m, n, c*} = A_{m, n} :: \lll_{c*} 1(ValAsCoord)
↔\\
Z_{m, n, B_{m, n}} = 1
\end{align}


Note that the *compute operator* will change.

In [ ]:
# New compute operator:
def set_one(coord, rhs_val):
  return 1


In [ ]:
Z = Tensor(rank_ids=["M", "N", "D*"])
Z.setName("Z")
#print(A.getShape())
#Z.setShape(A.getShape()+ A.get)
displayTensor(Z)
#Z.setDefault(None)
a_m = A.getRoot()
z_m = Z.getRoot()

# N = A.getShape("N")

# canvas = createCanvas(A, Z)

# Let's walk the iteration space:
for m, (z_n, a_n) in z_m << a_m:
  for n, (z_d, a_val) in z_n << a_n:
    ######################################
    # Now, this is the input to populate:
    ######################################
    # Get the current d fiber of Z -- we're 1D so it is just z_d
    populate(z_d, d, a_val, coord_op=ValAsCoord, compute_op=set_one)


print("***********************************")
displayTensor(A)
displayTensor(Z)

***
## Example 4: Largest-2 values and preserve coordinates and place in different fibers

We want to:
  1. pick the largest to values in some fiber
  2. place each of those two fibers in their own fiber such that they are next to each other.

Pick the m-th largest value for the $E$ rank:

\begin{align}
    TS0_{e*} = SO_{e} :: \lll_{e*} \text(pass-through) (\text{largest-2})\\
    TS2_{\rho(m), e} = TS0_{e} \cdot I_{m, e} :: \bigwedge ←
\end{align}

- place according to the position in the original fiber
- the $I$ tensor is the identity tensor
- $\rho(m)$ shifts the non-empty fibers such that their positions become their coordinates.

Example:
```Python
TSO_e = [0, 3, 0, 0, 7] --> coordinates (1, 4)
TS2_{m, e} = [[0, 3, 0, 0, 0],
              [0, 0, 0, 0, 7]]
```
- Why do we multiply by identity?
  - to get each element in the $e$ fiber into its own fiber
  - to create a 2D tensor that only has one element per row
    ```Python
  TS1_{m, e} = [[0, 0, 0, 0, 0],
                [0, 3, 0, 0, 0],
                ...
                [0, 0, 0, 0, 7]]
  ```
  - THEN we need to rearrange them --> using $\rho$
  \begin{align}
    TS2_{\rho(m), e} =  TS1_{m, e} ::
  \end{align}

- $\rho(m)$
  - gets the *position* of each non-empty $m$ coordinate
  - $\rho$ knows the sparsity but not the actual value of the payload
  

- rank variable expressions
  - any function that operates ONLY on rank variables
  - note that rank variable expressions on the output occur AFTER all maps, reduces, and populates
  - the assignment of values into the output variable happen after the creation of the populate temp

- populate
  - any function that operates on both coordinates and payloads


## Example 5: Example 4, but sorted from largest values to lowest values

As an example, if we have TS0 as an input, we want TS2 as an output:
```Python
TSO_e = [0, 3, 0, 0, 7] --> coordinates (1, 4)
TS2_{m, e} = [[0, 0, 0, 0, 7],
              [0, 3, 0, 0, 0]]
```

The steps for this are as follows:
1. Sort $TS0_e$ --> this is a separate Einsum on its own. For example bubble sort (in ascending order) is the following Einsum (more or less, needs a few more edits):
  \begin{align}
\text{initialization}\\
Z_{j, 0, n} = \begin{cases}
            min(A_{0}, A_{1}) & n=0 \\
            max(A_{0}, A_{1}) & n=1 \\
            0 & \text{o\w}
           \end{cases} \\
\text{Main EDGE}\\
i = 1 \\
j = 1 \\
Z_{i, j, n} = \begin{cases}
              Z_{i, j-1, n} & n < j\\
              min(Z_{i, j-1, j} \cdot Z_{i-1, j+1, j+1}) & n=j\\
              max(Z_{i, j-1, j} \cdot Z_{i-1, j+1, j+1}) & n=j+1\\
              0 & \text{o/w}
            \end{cases}\\
\diamond \diamond: j \equiv J\\
i = i+1\\
\diamond: i \equiv I
\end{align}

2. Let's call $RO$ the about of the sorted $TS0$ tensor:
 ```Python
RO_e = [0, 0, 0, 7, 3]
```

3. Now, we can apply our Einsum from example 4:
\begin{align}
    TSO_{e*} = SO_{e} :: \lll_{e*} \text(pass-through) (\text{largest-2})\\
    RO_{e} = \textbf{sortf}(TSO_e) \\
    TR1_{m, e} = RO_{e} \cdot I_{m, e}  :: \bigwedge ←
    TR2_{\rho(m), e} = TR1_{m, e}
\end{align}

 ```Python
TS2_{m, e} = [[0, 0, 0, 7, 0],
              [0, 0, 0, 0, 3]]
```

**Another Approach**:
1. Another approach is to first apply our Einsum from example 4:
\begin{align}
    TS0_{e*} = SO_{e} :: \lll_{e*} \text(pass-through) (\text{largest-2})\\
    TS2_{\rho(m), e} = TS0_{e} \cdot I_{m, e} :: \bigwedge ←
\end{align}


Note this would give us something like this:
```Python
TSO_e = [0, 3, 0, 0, 7] --> coordinates (1, 4)
TS2_{m, e} = [[0, 3, 0, 0, 0],
              [0, 0, 0, 0, 7]]
```

2. Now sort the output of the above Einsum
```Python
TS3_{m, e} = [[0, 7, 0, 0, 0],
              [0, 0, 0, 0, 3]]
```
\begin{align}
    TS0_{e*} &= SO_{e} :: \lll_{e*} \text(pass-through) (\text{largest-2})\\
    TS2_{\rho(m), e} &= TS0_{e} \cdot I_{m, e} :: \bigwedge ← \\
    TS3_{m, e} &= \textbf{sortf}(TS2_{m, e})
\end{align}


***
